# ROBUST04 LambdaMART Learning-to-Rank (LightGBM)

**Final Project Part A - Run 3 (Advanced Method)**

**What is LambdaMART?**
- Gradient boosting tree-based learning-to-rank
- Used in production search engines (**Microsoft Bing uses LightGBM!**)
- Directly optimizes for MAP 

**Why LightGBM?**
- 2-10x faster than XGBoost
- Native `lambdarank` objective (purpose-built for LTR)
- Lower memory usage
- Better MAP optimization

**Pipeline:**
1. **Stage 1**: BM25 retrieves candidate documents
2. **Feature Extraction**: Extract 12 features per query-doc pair
3. **Stage 2**: LambdaMART reranks using learned model

**Expected gain:** 10-20% over BM25 baseline

## 2. Install Dependencies

In [0]:
# Install LightGBM and Pyserini
%pip install lightgbm -q
%pip install pyserini==0.21.0 -q
# Install XGBoost and other dependencies

%pip install lightgbm -q
%pip install pyserini==0.21.0 -q
%pip install faiss-cpu -q #==1.7.4 -q
%pip install huggingface-hub #==0.20.0 -q
%pip install sentence-transformers #==2.2.2 -q

dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
petastorm 0.12.1 requires pyspark>=2.1.0, which is not installed.
databricks-feature-engineering 0.8.0 requires numpy<2,>=1.19.2, but you have numpy 2.4.1 which is incompatible.
langchain 0.1.20 requires numpy<2,>=1, but you have numpy 2.4.1 which is incompatible.
langchain-community 0.0.38 requires numpy<2,>=1, but you have numpy 

## 1. Configuration

In [0]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# File paths
QUERIES_PATH = "queriesROBUST.txt"
QRELS_PATH = "qrels_50_Queries"
OUTPUT_DIR = "runs/"

# BM25 parameters (for initial retrieval)
BM25_K1_DEFAULT = 0.6
BM25_B_DEFAULT = 0.4

# Feature extraction - multiple BM25/QLD variants
BM25_VARIANTS = [
    (0.6, 0.4),  # Tuned params from Run 1
    (0.9, 0.4),  # Higher k1
    (1.2, 0.75), # TREC defaults
]
QLD_MU_VARIANTS = [1000, 1500, 2000]

# LambdaMART parameters (LightGBM)
LAMBDAMART_PARAMS = {
    'objective': 'lambdarank',   # Native LambdaMART objective
    'metric': 'map',              # Optimize for MAP
    'learning_rate': 0.1,
    'num_leaves': 31,            # LightGBM uses leaves instead of depth
    'max_depth': 6,
    'min_data_in_leaf': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 0.0,
    'lambda_l2': 0.0,
    'verbose': -1
}
NUM_BOOST_ROUND = 150

# Retrieval
RETRIEVAL_K = 1000
FINAL_K = 1000
RUN_TAG = "lambdamart"

print("✅ Configuration loaded")

✅ Configuration loaded


## 3. Setup & Imports

In [0]:
import subprocess
import os

# Set Java environment
os.environ['JAVA_HOME'] = '/usr/lib/jvm/zulu17-ca-amd64'
os.environ['JVM_PATH'] = '/usr/lib/jvm/zulu17-ca-amd64/lib/server/libjvm.so'

# Imports
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
from collections import defaultdict
from tqdm import tqdm
import pandas as pd
import numpy as np
import json
import math
import lightgbm as lgb

print("✅ Imports complete")

2026-01-22 21:20:33.391700: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-22 21:20:34.705466: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


✅ Imports complete


## 4. Load Index & Data

In [0]:
# Load ROBUST04 index
print("Loading ROBUST04 index...")
searcher = LuceneSearcher.from_prebuilt_index('robust04')
index_reader = IndexReader.from_prebuilt_index('robust04')
print(f"✅ Index loaded: {index_reader.stats()}")

# Load queries
def load_queries(queries_path):
    queries = {}
    with open(queries_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t', 1)
            if len(parts) == 2:
                qid = parts[0].strip()
                query_text = parts[1].strip()
                queries[qid] = query_text
    return queries

# Load qrels
def load_qrels(qrels_path):
    qrels = defaultdict(dict)
    with open(qrels_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid = parts[0]
                doc_id = parts[2]
                relevance = int(parts[3])
                qrels[qid][doc_id] = relevance
    return qrels

all_queries = load_queries(QUERIES_PATH)
qrels = load_qrels(QRELS_PATH)
train_qids = list(qrels.keys())

train_queries = {qid: all_queries[qid] for qid in train_qids if qid in all_queries}
test_qids = [qid for qid in all_queries.keys() if qid not in train_qids]
test_queries = {qid: all_queries[qid] for qid in test_qids}

print(f"✅ Queries: {len(train_queries)} train, {len(test_queries)} test")

Loading ROBUST04 index...


lucene-index.robust04.20221005.252b5e.tar.gz: 1.68GB [00:13, 138MB/s]                            


✅ Index loaded: {'total_terms': 174540872, 'documents': 528030, 'non_empty_documents': 528030, 'unique_terms': 923436}
✅ Queries: 50 train, 199 test


## 5. Feature Extraction Functions

In [0]:
def get_bm25_score(query, doc_id, k1, b):
    """Get BM25 score for a specific query-doc pair."""
    searcher.set_bm25(k1=k1, b=b)
    hits = searcher.search(query, k=1000)
    
    for hit in hits:
        if hit.docid == doc_id:
            return hit.score
    
    return 0.0  # Document not in top-1000

def get_qld_score(query, doc_id, mu):
    """Get QLD score for a specific query-doc pair."""
    searcher.set_qld(mu=mu)
    hits = searcher.search(query, k=1000)
    
    for hit in hits:
        if hit.docid == doc_id:
            return hit.score
    
    return 0.0

def get_doc_length(doc_id):
    """Get document length."""
    try:
        doc_vector = index_reader.get_document_vector(doc_id)
        if doc_vector:
            return sum(doc_vector.values())
        return 0
    except:
        return 0

def get_term_frequency(doc_id, term):
    """Get term frequency in document."""
    try:
        doc_vector = index_reader.get_document_vector(doc_id)
        if doc_vector and term in doc_vector:
            return doc_vector[term]
        return 0
    except:
        return 0

def compute_covered_query_terms(query, doc_id):
    """Compute ratio of query terms that appear in document."""
    query_terms = query.lower().split()
    if not query_terms:
        return 0.0
    
    covered = 0
    for term in query_terms:
        if get_term_frequency(doc_id, term) > 0:
            covered += 1
    
    return covered / len(query_terms)

def compute_sum_tf(query, doc_id):
    """Sum of term frequencies for query terms in document."""
    query_terms = query.lower().split()
    return sum(get_term_frequency(doc_id, term) for term in query_terms)

def extract_features(query, doc_id, initial_rank=None):
    """
    Extract all features for a query-document pair.
    
    Returns 12-dimensional feature vector.
    """
    features = []
    
    # Feature 1-3: BM25 with different parameters
    for k1, b in BM25_VARIANTS:
        features.append(get_bm25_score(query, doc_id, k1, b))
    
    # Feature 4-6: QLD with different mu values
    for mu in QLD_MU_VARIANTS:
        features.append(get_qld_score(query, doc_id, mu))
    
    # Feature 7: Query length
    features.append(len(query.split()))
    
    # Feature 8: Document length (log scale)
    doc_len = get_doc_length(doc_id)
    features.append(math.log(doc_len + 1))
    
    # Feature 9: Covered query term ratio
    features.append(compute_covered_query_terms(query, doc_id))
    
    # Feature 10: Sum of term frequencies
    features.append(compute_sum_tf(query, doc_id))
    
    # Feature 11: Initial ranking position (if available)
    if initial_rank is not None:
        features.append(1.0 / (initial_rank + 1))  # Reciprocal rank
    else:
        features.append(0.0)
    
    # Feature 12: Query-doc length ratio
    query_len = len(query.split())
    if doc_len > 0:
        features.append(query_len / doc_len)
    else:
        features.append(0.0)
    
    return features

print("✅ Feature extraction functions ready")
print(f"   Total features: 12")
print(f"   - BM25 variants: {len(BM25_VARIANTS)}")
print(f"   - QLD variants: {len(QLD_MU_VARIANTS)}")
print(f"   - Other features: 6")

✅ Feature extraction functions ready
   Total features: 12
   - BM25 variants: 3
   - QLD variants: 3
   - Other features: 6


## 6. Create Training Dataset

In [0]:
print("🔨 Creating training dataset...")
print("   This will take 10-15 minutes (extracting features for all query-doc pairs)\n")

X_train = []
y_train = []
qids_train = []
group_sizes = []

# Process each training query
for qid in tqdm(sorted(train_queries.keys()), desc="Processing queries"):
    query = train_queries[qid]
    
    # Get initial BM25 ranking
    searcher.set_bm25(k1=BM25_K1_DEFAULT, b=BM25_B_DEFAULT)
    hits = searcher.search(query, k=RETRIEVAL_K)
    
    if not hits:
        continue
    
    query_features = []
    query_labels = []
    
    # Extract features for each retrieved document
    for rank, hit in enumerate(hits):
        doc_id = hit.docid
        
        # Extract features
        features = extract_features(query, doc_id, initial_rank=rank)
        
        # Get label from qrels (0 or 1)
        label = qrels[qid].get(doc_id, 0)
        
        query_features.append(features)
        query_labels.append(label)
    
    # Add to training set
    X_train.extend(query_features)
    y_train.extend(query_labels)
    qids_train.extend([qid] * len(query_features))
    group_sizes.append(len(query_features))

# Convert to numpy arrays
X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"\n✅ Training dataset created:")
print(f"   Samples: {len(X_train):,}")
print(f"   Features: {X_train.shape[1]}")
print(f"   Queries: {len(group_sizes)}")
print(f"   Positive samples: {sum(y_train):,} ({sum(y_train)/len(y_train)*100:.1f}%)")

🔨 Creating training dataset...
   This will take 10-15 minutes (extracting features for all query-doc pairs)



Processing queries:  66%|██████▌   | 33/50 [4:22:57<2:36:38, 552.84s/it]

## 7. Train LambdaMART Model (LightGBM)

In [0]:
print("🚀 Training LambdaMART model with LightGBM...\n")

# Create LightGBM Dataset with group information
dtrain = lgb.Dataset(X_train, label=y_train, group=group_sizes)

# Train model
model = lgb.train(
    LAMBDAMART_PARAMS,
    dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[dtrain],
    valid_names=['train'],
    callbacks=[lgb.log_evaluation(period=20)]
)

print("\n✅ LambdaMART model trained with LightGBM!")
print(f"   Training completed in {NUM_BOOST_ROUND} iterations")
print(f"   Final MAP: Check above")

## 8. Feature Importance Analysis

In [0]:
# Get feature importance
importance = model.feature_importance(importance_type='gain')

feature_names = (
    [f"BM25(k1={k1},b={b})" for k1, b in BM25_VARIANTS] +
    [f"QLD(mu={mu})" for mu in QLD_MU_VARIANTS] +
    ["QueryLength", "DocLength(log)", "CoveredTerms", "SumTF", 
     "ReciprocalRank", "QueryDocLenRatio"]
)

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance
}).sort_values('Importance', ascending=False)

print("📊 Feature Importance (Top 10):")
print("   (Higher importance = more useful for ranking)")
display(importance_df.head(10))

## 9. Validation on Train Set

In [0]:
print("🧪 Validating LambdaMART on train set...")

def rerank_with_lambdamart(query, model, k=1000):
    """Rerank documents using trained LambdaMART model."""
    # Get initial BM25 ranking
    searcher.set_bm25(k1=BM25_K1_DEFAULT, b=BM25_B_DEFAULT)
    hits = searcher.search(query, k=k)
    
    if not hits:
        return []
    
    # Extract features for all docs
    X_test = []
    doc_ids = []
    
    for rank, hit in enumerate(hits):
        features = extract_features(query, hit.docid, initial_rank=rank)
        X_test.append(features)
        doc_ids.append(hit.docid)
    
    # Predict scores with LightGBM
    scores = model.predict(np.array(X_test))
    
    # Combine and sort
    results = list(zip(doc_ids, scores))
    results.sort(key=lambda x: x[1], reverse=True)
    
    return results

# Test on train queries
val_results = {}

for qid, query in tqdm(train_queries.items(), desc="Validating"):
    val_results[qid] = rerank_with_lambdamart(query, model, k=RETRIEVAL_K)

# Calculate MAP
def calculate_ap(ranked_list, qrels_for_query):
    """Calculate Average Precision."""
    if not qrels_for_query:
        return 0.0
    
    num_relevant = sum(1 for rel in qrels_for_query.values() if rel > 0)
    if num_relevant == 0:
        return 0.0
    
    num_relevant_retrieved = 0
    precision_sum = 0.0
    
    for i, doc_id in enumerate(ranked_list, start=1):
        if doc_id in qrels_for_query and qrels_for_query[doc_id] > 0:
            num_relevant_retrieved += 1
            precision_at_i = num_relevant_retrieved / i
            precision_sum += precision_at_i
    
    if num_relevant_retrieved == 0:
        return 0.0
    
    return precision_sum / num_relevant

def calculate_map(retrieval_results, qrels):
    """Calculate Mean Average Precision."""
    ap_scores = []
    
    for qid in retrieval_results:
        if qid not in qrels:
            continue
        
        ranked_list = [doc_id for doc_id, _ in retrieval_results[qid]]
        ap = calculate_ap(ranked_list, qrels[qid])
        ap_scores.append(ap)
    
    if not ap_scores:
        return 0.0
    
    return sum(ap_scores) / len(ap_scores)

val_map = calculate_map(val_results, qrels)

print(f"\n{'='*60}")
print(f"📊 VALIDATION RESULTS")
print(f"{'='*60}")
print(f"LambdaMART MAP on train: {val_map:.4f}")
print(f"\nComparison:")
print(f"  BM25 baseline: 0.2475")
print(f"  BM25 + RM3: 0.2713")
print(f"  LambdaMART: {val_map:.4f}")
print(f"  Gain over BM25: {((val_map - 0.2475) / 0.2475 * 100):+.2f}%")
print(f"{'='*60}")

## 10. Generate Run 3 - Full Test Set

In [0]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"🚀 Generating Run 3: LambdaMART Learning-to-Rank")
print(f"   Processing all {len(all_queries)} queries...\n")

all_results = {}

for qid, query in tqdm(all_queries.items(), desc="Run 3"):
    all_results[qid] = rerank_with_lambdamart(query, model, k=RETRIEVAL_K)

# Save TREC format
def save_trec_format(retrieval_results, output_path, run_tag):
    with open(output_path, 'w') as f:
        for qid in sorted(retrieval_results.keys(), key=int):
            ranked_docs = retrieval_results[qid]
            for rank, (doc_id, score) in enumerate(ranked_docs, start=1):
                line = f"{qid} Q0 {doc_id} {rank} {score:.6f} {run_tag}\n"
                f.write(line)
    print(f"✅ Saved: {output_path}")

run3_path = os.path.join(OUTPUT_DIR, "run_3.res")
save_trec_format(all_results, run3_path, RUN_TAG)

print(f"\n📊 Run 3 Statistics:")
print(f"   Total queries: {len(all_results)}")
print(f"   Avg docs per query: {sum(len(docs) for docs in all_results.values()) / len(all_results):.1f}")

# Show sample
print(f"\n🔍 Sample output (first 5 lines):")
with open(run3_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(f"   {line.strip()}")

## 11. Per-Query Analysis

In [0]:
# Analyze performance per query
train_results_final = {qid: all_results[qid] for qid in train_queries.keys()}
train_map_final = calculate_map(train_results_final, qrels)

per_query_analysis = []

for qid in train_queries.keys():
    ranked_list = [doc_id for doc_id, _ in train_results_final[qid]]
    ap = calculate_ap(ranked_list, qrels[qid])
    
    per_query_analysis.append({
        'query_id': qid,
        'query_text': train_queries[qid],
        'AP': ap,
        'num_relevant': sum(1 for rel in qrels[qid].values() if rel > 0)
    })

df_analysis = pd.DataFrame(per_query_analysis).sort_values('AP')

print("📊 Per-Query Analysis: LambdaMART\n")
print("Worst 5 performing queries:")
display(df_analysis.head(5)[['query_id', 'query_text', 'AP']])

print("\nBest 5 performing queries:")
display(df_analysis.tail(5)[['query_id', 'query_text', 'AP']])

print(f"\nAP Statistics:")
print(df_analysis['AP'].describe())

## 12. Summary

In [0]:
print(f"""
{'='*60}
🎉 LAMBDAMART LEARNING-TO-RANK COMPLETE (LightGBM)!
{'='*60}

📊 Final Results Summary:

Method: LambdaMART (LightGBM - Microsoft Bing's Algorithm)
   • Features: 12 (BM25 variants, QLD variants, query/doc stats)
   • Training: {NUM_BOOST_ROUND} boosting rounds
   • Train MAP: {train_map_final:.4f}

Performance Comparison:
   • BM25 baseline: 0.2475
   • BM25 + RM3: 0.2713
   • LambdaMART: {train_map_final:.4f}
   • Gain over BM25: {((train_map_final - 0.2475) / 0.2475 * 100):+.2f}%
   • Gain over RM3: {((train_map_final - 0.2713) / 0.2713 * 100):+.2f}%

📁 Output:
   • {run3_path}

🎓 Why LightGBM + LambdaMART?
   1. Production-grade: Microsoft Bing uses LightGBM!
   2. Native lambdarank: Purpose-built for ranking
   3. 2-10x faster than XGBoost
   4. Directly optimizes MAP: Your evaluation metric
   5. Feature engineering: Shows deep IR understanding
   6. Interpretable: Can analyze feature importance

🔬 Key Innovation:
   Instead of hand-tuned scoring, LambdaMART LEARNS
   optimal feature combinations from training data.
   
   This is "learning-to-rank" - the model learns
   what makes documents relevant based on examples!

🚀 Why LightGBM specifically:
   • Microsoft developed it for Bing search
   • Faster and more memory-efficient than XGBoost
   • Native support for ranking objectives
   • Better gradient-based optimization for MAP

📈 Your 3 Submissions:
   ✅ Run 1: BM25 (tuned parameters)
   ✅ Run 2: BM25 + RM3 (query expansion) 
   ✅ Run 3: LambdaMART/LightGBM (ML learning-to-rank)

💡 Presentation Talking Points:
   • Classical → Expansion → Machine Learning progression
   • Feature engineering demonstrates IR expertise
   • LightGBM is Bing's actual ranking algorithm!
   • Directly optimizes the evaluation metric
   • Shows understanding of production systems

{'='*60}
""")

## 13. Save Model 

In [0]:
# Save LightGBM model for future use
model_path = os.path.join(OUTPUT_DIR, "lambdamart_lgb_model.txt")
model.save_model(model_path)
print(f"💾 Model saved: {model_path}")
print(f"   Model size: ~{os.path.getsize(model_path) / 1024:.1f} KB")

## 14. Hyperparameter Tuning 

**Skip this if you're happy with results. Run only if you want to squeeze more performance.**

In [0]:
# Grid search over LambdaMART hyperparameters
print("🔍 Hyperparameter tuning (optional)...")

param_grid = {
    'learning_rate': [0.05, 0.1, 0.15],
    'num_leaves': [15, 31, 63],
}

best_map = 0
best_params = None

from itertools import product

for lr, num_leaves in product(param_grid['learning_rate'], param_grid['num_leaves']):
    params = LAMBDAMART_PARAMS.copy()
    params['learning_rate'] = lr
    params['num_leaves'] = num_leaves
    
    # Train
    model_temp = lgb.train(
        params, 
        dtrain, 
        num_boost_round=100,
        valid_sets=[dtrain],
        callbacks=[lgb.log_evaluation(period=0)]  # Silent
    )
    
    # Validate
    val_temp = {}
    for qid, query in train_queries.items():
        val_temp[qid] = rerank_with_lambdamart(query, model_temp, k=RETRIEVAL_K)
    
    map_score = calculate_map(val_temp, qrels)
    
    print(f"lr={lr}, num_leaves={num_leaves}: MAP={map_score:.4f}")
    
    if map_score > best_map:
        best_map = map_score
        best_params = (lr, num_leaves)

print(f"\n✅ Best params: lr={best_params[0]}, num_leaves={best_params[1]}")
print(f"   Best MAP: {best_map:.4f}")

## 15. Cleanup

In [0]:
print("✅ LambdaMART Run 3 complete!")
print("\n🎯 Key Advantages of LightGBM over XGBoost:")
print("   • 2-10x faster training")
print("   • Lower memory usage")
print("   • Native lambdarank objective")
print("   • Used by Microsoft Bing")
print("   • Better MAP optimization")